Install Libraries

In [ ]:
!pip install xgboost shap -q

Import Libraries

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

import matplotlib.pyplot as plt
import shap

Load Dataset

In [ ]:
df = pd.read_csv('/content/Nepal_Landslide_Training_Final.csv')

print(df.shape)
print(df.head())

Remove Unnecessary Columns

In [ ]:
df = df.drop(columns=[
    'system:index',
    '.geo'
])

print(df.columns)

Check Class Distribution

In [ ]:
print(df['Class'].value_counts())

Convert Aspect to Circular Variables

In [ ]:
df['Aspect_sin'] = np.sin(
    np.radians(df['Aspect'])
)

df['Aspect_cos'] = np.cos(
    np.radians(df['Aspect'])
)

df.drop('Aspect', axis=1, inplace=True)

One-Hot Encode LULC

In [ ]:
df['LULC'] = df['LULC'].astype(str)

df = pd.get_dummies(
    df,
    columns=['LULC']
)

print(df.shape)

Define Predictors and Target

In [ ]:
X = df.drop('Class', axis=1)

y = df['Class']

print(X.columns)

Train-Test Split (70/30)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.30,

    stratify=y,

    random_state=42
)

print(X_train.shape)
print(X_test.shape)

Random Forest Baseline

In [ ]:
rf = RandomForestClassifier(

    n_estimators=500,

    max_depth=15,

    random_state=42,

    n_jobs=-1
)

rf.fit(
    X_train,
    y_train
)

Random Forest Predictions

In [ ]:
rf_pred = rf.predict(X_test)

rf_prob = rf.predict_proba(X_test)[:,1]

Random Forest Evaluation

In [ ]:
rf_acc = accuracy_score(
    y_test,
    rf_pred
)

rf_auc = roc_auc_score(
    y_test,
    rf_prob
)

print("Random Forest Accuracy:", rf_acc)
print("Random Forest AUC:", rf_auc)

print(
    classification_report(
        y_test,
        rf_pred
    )
)

Random Forest Confusion Matrix

In [ ]:
cm = confusion_matrix(
    y_test,
    rf_pred
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm
)

disp.plot(
    cmap='Blues'
)

plt.title(
    'Random Forest Confusion Matrix'
)

plt.show()

Random Forest Feature Importance

In [ ]:
rf_importance = pd.DataFrame({

    'Feature':X.columns,

    'Importance':
    rf.feature_importances_

})

rf_importance = rf_importance.sort_values(

    by='Importance',

    ascending=False
)

print(
    rf_importance.head(20)
)

Plot RF Importance

In [ ]:
plt.figure(figsize=(10,8))

plt.barh(
    rf_importance['Feature'][:15],
    rf_importance['Importance'][:15]
)

plt.gca().invert_yaxis()

plt.title(
    'Random Forest Feature Importance'
)

plt.show()

XGBoost Model

In [ ]:
xgb = XGBClassifier(

    n_estimators=500,

    learning_rate=0.05,

    max_depth=6,

    subsample=0.8,

    colsample_bytree=0.8,

    random_state=42,

    eval_metric='logloss'
)

xgb.fit(
    X_train,
    y_train
)

XGBoost Predictions

In [ ]:
xgb_pred = xgb.predict(
    X_test
)

xgb_prob = xgb.predict_proba(
    X_test
)[:,1]

XGBoost Evaluation

In [ ]:
xgb_acc = accuracy_score(
    y_test,
    xgb_pred
)

xgb_auc = roc_auc_score(
    y_test,
    xgb_prob
)

print("XGBoost Accuracy:", xgb_acc)
print("XGBoost AUC:", xgb_auc)

print(
    classification_report(
        y_test,
        xgb_pred
    )
)

XGBoost Confusion Matrix

In [ ]:
cm = confusion_matrix(
    y_test,
    xgb_pred
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm
)

disp.plot(
    cmap='Greens'
)

plt.title(
    'XGBoost Confusion Matrix'
)

plt.show()

Compare Models

In [ ]:
comparison = pd.DataFrame({

    'Model':['Random Forest','XGBoost'],

    'Accuracy':[rf_acc,xgb_acc],

    'ROC_AUC':[rf_auc,xgb_auc]
})

print(comparison)

SHAP Analysis (XGBoost)

In [ ]:
explainer = shap.TreeExplainer(
    xgb
)

shap_values = explainer.shap_values(
    X_test
)

SHAP Global Importance

In [ ]:
shap.summary_plot(
    shap_values,
    X_test
)

SHAP Bar Plot

In [ ]:
shap.summary_plot(

    shap_values,

    X_test,

    plot_type='bar'
)


SHAP Dependence Plot

In [ ]:
shap.dependence_plot(
    'Slope',
    shap_values,
    X_test
)

In [ ]:
shap.dependence_plot(
    'Elevation',
    shap_values,
    X_test
)

In [ ]:
shap.dependence_plot(
    'Rainfall',
    shap_values,
    X_test
)

In [ ]:
shap.dependence_plot(
    'NDVI',
    shap_values,
    X_test
)

Save Feature Importance

In [ ]:
rf_importance.to_csv(
    '/content/RF_Feature_Importance.csv',
    index=False
)

Save Trained Models

In [ ]:
import joblib

joblib.dump(
    rf,
    '/content/Nepal_RF_Model.pkl'
)

joblib.dump(
    xgb,
    '/content/Nepal_XGB_Model.pkl'
)

 Add 10-Fold Cross Validation

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    rf,
    X,
    y,
    cv=10,
    scoring='roc_auc'
)

print(scores.mean())

Add Cohen's Kappa

In [ ]:
from sklearn.metrics import cohen_kappa_score

kappa = cohen_kappa_score(
    y_test,
    xgb_pred
)


Add Matthews Correlation Coefficient

In [ ]:
from sklearn.metrics import matthews_corrcoef

mcc = matthews_corrcoef(
    y_test,
    xgb_pred
)

Create ROC Comparison Figure

In [ ]:
from sklearn.metrics import roc_curve

In [ ]:
from sklearn.model_selection import cross_val_score

cv_auc = cross_val_score(
    xgb,
    X,
    y,
    cv=10,
    scoring='roc_auc',
    n_jobs=-1
)

print("Mean AUC:", cv_auc.mean())
print("Std AUC:", cv_auc.std())

In [ ]:
from sklearn.metrics import cohen_kappa_score
from sklearn.metrics import matthews_corrcoef

kappa = cohen_kappa_score(y_test, xgb_pred)
mcc = matthews_corrcoef(y_test, xgb_pred)

print("Kappa:", kappa)
print("MCC:", mcc)

Figure 3

In [ ]:
from sklearn.metrics import roc_curve

rf_fpr, rf_tpr, _ = roc_curve(
    y_test,
    rf_prob
)

xgb_fpr, xgb_tpr, _ = roc_curve(
    y_test,
    xgb_prob
)

In [ ]:
import matplotlib.pyplot as plt
import shap

fig = plt.figure(
    figsize=(16,12)
)

# ====================================
# (a) ROC Curve
# ====================================

ax1 = plt.subplot(2,2,1)

ax1.plot(
    rf_fpr,
    rf_tpr,
    linewidth=2,
    label=f'RF (AUC={rf_auc:.3f})'
)

ax1.plot(
    xgb_fpr,
    xgb_tpr,
    linewidth=2,
    label=f'XGB (AUC={xgb_auc:.3f})'
)

ax1.plot(
    [0,1],
    [0,1],
    '--',
    color='gray'
)

ax1.set_title('(a) ROC Curve Comparison')
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.legend()

# ====================================
# (b) RF Feature Importance
# ====================================

ax2 = plt.subplot(2,2,2)

top10 = rf_importance.head(10)

ax2.barh(
    top10['Feature'][::-1],
    top10['Importance'][::-1]
)

ax2.set_title('(b) RF Feature Importance')

# ====================================
# SAVE TEMP FIGURE
# ====================================

plt.tight_layout()

plt.savefig(
    '/content/Figure4_ab.png',
    dpi=600,
    bbox_inches='tight'
)

plt.show()

In [ ]:
import shap
import matplotlib.pyplot as plt

# ----------------------
# SHAP Summary
# ----------------------

shap.summary_plot(
    shap_values,
    X_test,
    show=False
)

plt.savefig(
    '/content/shap_summary.png',
    dpi=600,
    bbox_inches='tight'
)

plt.close()

# ----------------------
# SHAP Bar
# ----------------------

shap.summary_plot(
    shap_values,
    X_test,
    plot_type='bar',
    show=False
)

plt.savefig(
    '/content/shap_bar.png',
    dpi=600,
    bbox_inches='tight'
)

plt.close()

In [ ]:
from sklearn.metrics import roc_curve
from PIL import Image

# ROC data
rf_fpr, rf_tpr, _ = roc_curve(
    y_test,
    rf_prob
)

xgb_fpr, xgb_tpr, _ = roc_curve(
    y_test,
    xgb_prob
)

# Read SHAP figures
shap_summary = Image.open(
    "/content/shap_summary.png"
)

shap_bar = Image.open(
    "/content/shap_bar.png"
)

# =====================================================
# 2 x 2 PANEL FIGURE
# =====================================================

fig, axes = plt.subplots(
    2,
    2,
    figsize=(16,12)
)

# -----------------------------------------------------
# (a) ROC Curve
# -----------------------------------------------------

axes[0,0].plot(
    rf_fpr,
    rf_tpr,
    linewidth=2,
    label=f'RF (AUC={rf_auc:.3f})'
)

axes[0,0].plot(
    xgb_fpr,
    xgb_tpr,
    linewidth=2,
    label=f'XGB (AUC={xgb_auc:.3f})'
)

axes[0,0].plot(
    [0,1],[0,1],
    '--',
    color='gray'
)

axes[0,0].set_title(
    '(a) ROC Curve Comparison',
    fontsize=12,
    fontweight='bold'
)

axes[0,0].set_xlabel(
    'False Positive Rate'
)

axes[0,0].set_ylabel(
    'True Positive Rate'
)

axes[0,0].legend()

# -----------------------------------------------------
# (b) RF Importance
# -----------------------------------------------------

top10 = rf_importance.head(10)

axes[0,1].barh(
    top10['Feature'][::-1],
    top10['Importance'][::-1],
    color='steelblue'
)

axes[0,1].set_title(
    '(b) RF Feature Importance',
    fontsize=12,
    fontweight='bold'
)

# -----------------------------------------------------
# (c) SHAP Summary
# -----------------------------------------------------

axes[1,0].imshow(
    shap_summary
)

axes[1,0].axis('off')

axes[1,0].set_title(
    '(c) SHAP Summary Plot',
    fontsize=12,
    fontweight='bold'
)

# -----------------------------------------------------
# (d) SHAP Bar
# -----------------------------------------------------

axes[1,1].imshow(
    shap_bar
)

axes[1,1].axis('off')

axes[1,1].set_title(
    '(d) SHAP Global Importance',
    fontsize=12,
    fontweight='bold'
)

plt.tight_layout()

plt.savefig(
    '/content/Figure4_ML_SHAP.png',
    dpi=600,
    bbox_inches='tight'
)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import roc_curve

from PIL import Image

from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D

# ======================================================
# SETTINGS
# ======================================================

plt.rcParams['font.family'] = 'Arial'
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10

# ======================================================
# COLORS
# ======================================================

RF_COLOR = "#1f77b4"
XGB_COLOR = "#d62728"

IMPORTANCE_COLOR = "#2c7fb8"

BORDER_COLOR = "black"

# ======================================================
# ROC DATA
# ======================================================

rf_fpr, rf_tpr, _ = roc_curve(
    y_test,
    rf_prob
)

xgb_fpr, xgb_tpr, _ = roc_curve(
    y_test,
    xgb_prob
)

# ======================================================
# LOAD SHAP IMAGES
# ======================================================

shap_summary = Image.open(
    "/content/shap_summary.png"
)

shap_bar = Image.open(
    "/content/shap_bar.png"
)

# ======================================================
# FIGURE
# ======================================================

fig, axes = plt.subplots(
    2,
    2,
    figsize=(18, 14),
    facecolor="white"
)

# ======================================================
# (a) ROC CURVE
# ======================================================

axes[0,0].plot(
    rf_fpr,
    rf_tpr,
    color=RF_COLOR,
    lw=3,
    label=f'Random Forest (AUC={rf_auc:.3f})'
)

axes[0,0].plot(
    xgb_fpr,
    xgb_tpr,
    color=XGB_COLOR,
    lw=3,
    label=f'XGBoost (AUC={xgb_auc:.3f})'
)

axes[0,0].plot(
    [0,1],
    [0,1],
    '--',
    color='gray',
    lw=1.5
)

axes[0,0].set_xlabel(
    'False Positive Rate'
)

axes[0,0].set_ylabel(
    'True Positive Rate'
)

axes[0,0].set_title(
    '(a) ROC Curve Comparison',
    fontsize=15,
    fontweight='bold'
)

axes[0,0].legend(
    frameon=True,
    fontsize=10
)

axes[0,0].grid(
    alpha=0.3
)

# ======================================================
# (b) RF IMPORTANCE
# ======================================================

top10 = rf_importance.head(10)

colors = plt.cm.Blues(
    np.linspace(
        0.45,
        0.90,
        len(top10)
    )
)

axes[0,1].barh(
    top10['Feature'][::-1],
    top10['Importance'][::-1],
    color=colors
)

for i, val in enumerate(
    top10['Importance'][::-1]
):
    axes[0,1].text(
        val + 0.004,
        i,
        f"{val:.3f}",
        fontsize=9,
        va='center'
    )

axes[0,1].set_title(
    '(b) Random Forest Feature Importance',
    fontsize=15,
    fontweight='bold'
)

axes[0,1].set_xlabel(
    'Importance Score'
)

# ======================================================
# (c) SHAP SUMMARY
# ======================================================

axes[1,0].imshow(
    shap_summary
)

axes[1,0].set_title(
    '(c) SHAP Summary Plot',
    fontsize=15,
    fontweight='bold'
)

axes[1,0].axis('off')

# ======================================================
# (d) SHAP BAR
# ======================================================

axes[1,1].imshow(
    shap_bar
)

axes[1,1].set_title(
    '(d) SHAP Global Importance',
    fontsize=15,
    fontweight='bold'
)

axes[1,1].axis('off')

# ======================================================
# PANEL BORDERS
# ======================================================

for ax in axes.flat:

    ax.set_facecolor('white')

    for spine in ax.spines.values():

        spine.set_visible(True)

        spine.set_linewidth(1.5)

        spine.set_color(BORDER_COLOR)

# ======================================================
# CUSTOM LEGEND (FIGURE LEVEL)
# ======================================================

legend_elements = [

    Line2D(
        [0],
        [0],
        color=RF_COLOR,
        lw=3,
        label='Random Forest'
    ),

    Line2D(
        [0],
        [0],
        color=XGB_COLOR,
        lw=3,
        label='XGBoost'
    ),

    Rectangle(
        (0,0),
        1,
        1,
        facecolor=IMPORTANCE_COLOR,
        label='Feature Importance'
    )

]

fig.legend(
    handles=legend_elements,
    loc='upper center',
    ncol=3,
    bbox_to_anchor=(0.5,0.98),
    frameon=True,
    fontsize=11
)


# ======================================================
# OUTER BORDER AROUND ENTIRE FIGURE
# ======================================================

outer_border = Rectangle(
    (0.005,0.005),
    0.99,
    0.99,

    fill=False,

    transform=fig.transFigure,

    figure=fig,

    linewidth=2.5,

    edgecolor='black'
)

fig.patches.append(
    outer_border
)

# ======================================================
# SPACING
# ======================================================

plt.tight_layout(
    rect=[
        0.02,
        0.02,
        0.98,
        0.95
    ]
)

# ======================================================
# EXPORT
# ======================================================

plt.savefig(
    '/content/Figure4_ML_SHAP_Q1.png',
    dpi=800,
    bbox_inches='tight',
    facecolor='white'
)

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

from sklearn.metrics import roc_curve

from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D

# =====================================================
# FONT SETTINGS
# =====================================================

plt.rcParams['font.family'] = 'DejaVu Sans'

plt.rcParams['axes.titlesize'] = 15
plt.rcParams['axes.titleweight'] = 'bold'

plt.rcParams['axes.labelsize'] = 12

plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10

# =====================================================
# CONSISTENT COLOR SCHEME
# =====================================================

RF_COLOR  = "#1f77b4"
XGB_COLOR = "#d62728"

# =====================================================
# ROC DATA
# =====================================================

rf_fpr, rf_tpr, _ = roc_curve(
    y_test,
    rf_prob
)

xgb_fpr, xgb_tpr, _ = roc_curve(
    y_test,
    xgb_prob
)

# =====================================================
# TOP 10 SHAP FEATURES
# =====================================================

shap_importance = pd.DataFrame({

    'Feature':X_test.columns,

    'Importance':
    np.abs(shap_values).mean(axis=0)

})

shap_importance = shap_importance.sort_values(

    by='Importance',

    ascending=False

)

shap_top10 = shap_importance.head(10)

# =====================================================
# LOAD SHAP SUMMARY IMAGE
# =====================================================

shap_summary = Image.open(
    '/content/shap_summary.png'
)

# =====================================================
# FIGURE
# =====================================================

fig, axes = plt.subplots(

    2,
    2,

    figsize=(18,14),

    facecolor='white'

)

# =====================================================
# (A) ROC CURVE
# =====================================================

axes[0,0].plot(

    rf_fpr,
    rf_tpr,

    color=RF_COLOR,

    linewidth=3,

    label=f'Random Forest (AUC={rf_auc:.3f})'

)

axes[0,0].plot(

    xgb_fpr,
    xgb_tpr,

    color=XGB_COLOR,

    linewidth=3,

    label=f'XGBoost (AUC={xgb_auc:.3f})'

)

axes[0,0].plot(

    [0,1],
    [0,1],

    '--',

    color='gray',

    linewidth=1.5

)

axes[0,0].set_title(
    '(a) ROC Curve Comparison'
)

axes[0,0].set_xlabel(
    'False Positive Rate'
)

axes[0,0].set_ylabel(
    'True Positive Rate'
)

axes[0,0].grid(
    alpha=0.3
)

axes[0,0].legend(
    frameon=True,
    fontsize=10
)

# =====================================================
# (B) RF FEATURE IMPORTANCE
# =====================================================

top10 = rf_importance.head(10)

rf_colors = plt.cm.Blues(

    np.linspace(
        0.45,
        0.90,
        len(top10)
    )

)

axes[0,1].barh(

    top10['Feature'][::-1],

    top10['Importance'][::-1],

    color=rf_colors

)

for i, v in enumerate(

    top10['Importance'][::-1]

):

    axes[0,1].text(

        v + 0.003,

        i,

        f'{v:.3f}',

        va='center',

        fontsize=9

    )

axes[0,1].set_title(
    '(b) Random Forest Feature Importance'
)

axes[0,1].set_xlabel(
    'Importance Score'
)

# =====================================================
# (C) SHAP SUMMARY
# =====================================================

axes[1,0].imshow(
    shap_summary
)

axes[1,0].axis('off')

axes[1,0].set_title(
    '(c) SHAP Summary Plot'
)

# =====================================================
# (D) SHAP GLOBAL IMPORTANCE
# =====================================================

shap_top10_rev = shap_top10.iloc[::-1]

shap_colors = plt.cm.Reds(

    np.linspace(
        0.45,
        0.90,
        len(shap_top10_rev)
    )

)

axes[1,1].barh(

    shap_top10_rev['Feature'],

    shap_top10_rev['Importance'],

    color=shap_colors

)

for i, v in enumerate(

    shap_top10_rev['Importance']

):

    axes[1,1].text(

        v + 0.02,

        i,

        f'{v:.2f}',

        va='center',

        fontsize=9

    )

axes[1,1].set_title(
    '(d) SHAP Global Importance'
)

axes[1,1].set_xlabel(
    'Mean |SHAP Value|'
)

# =====================================================
# PANEL BORDERS
# =====================================================

for ax in axes.flat:

    ax.set_facecolor('white')

    for spine in ax.spines.values():

        spine.set_visible(True)

        spine.set_linewidth(1.5)

        spine.set_color('black')

# =====================================================
# CUSTOM FIGURE LEGEND
# =====================================================

legend_elements = [

    Line2D(

        [0],
        [0],

        color=RF_COLOR,

        linewidth=3,

        label='Random Forest'

    ),

    Line2D(

        [0],
        [0],

        color=XGB_COLOR,

        linewidth=3,

        label='XGBoost'

    ),

    Rectangle(

        (0,0),

        1,
        1,

        facecolor='#1f77b4',

        alpha=0.8,

        label='Feature Importance'

    ),

    Rectangle(

        (0,0),

        1,
        1,

        facecolor='#d62728',

        alpha=0.8,

        label='SHAP Importance'

    )

]

fig.legend(

    handles=legend_elements,

    loc='upper center',

    bbox_to_anchor=(0.5,0.98),

    ncol=4,

    frameon=True,

    fontsize=11

)



# =====================================================
# SPACING
# =====================================================

plt.tight_layout(

    rect=[
        0.02,
        0.02,
        0.98,
        0.95
    ]

)

# =====================================================
# SAVE
# =====================================================

plt.savefig(

    '/content/Figure4_ML_SHAP_Q1_Final.png',

    dpi=800,

    bbox_inches='tight',

    facecolor='white'

)

plt.show()

Study Area

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx

from matplotlib.patches import Patch
from matplotlib_scalebar.scalebar import ScaleBar
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# ======================================
# LOAD DATA
# ======================================

world = gpd.read_file(
'/content/ne_10m_admin_0_countries_nep.shp'
)

nepal = gpd.read_file(
'/content/1_NepalBoundary.shp'
)

districts = gpd.read_file(
'/content/gadm41_NPL_3.shp'
)

# ======================================
# LOAD LANDSLIDES
# ======================================

landslides = gpd.read_file(
'/content/ICIMOD_Landslides.shp'
)

# ======================================
# SELECT STUDY DISTRICTS
# ======================================

study_names = [

'Sindhupalchok',
'Sindhuli',
'Ramechhap',
'Okhaldhunga',
'Nuwakot',
'Makwanpur',
'Lalitpur',
'Kavrepalanchok',
'Kathmandu',
'Gorkha',
'Dolakha',
'Dhading',
'Rasuwa'

]

study_districts = districts[
districts['NAME_3'].isin(study_names)
]

# ======================================
# REPROJECT
# ======================================

world = world.to_crs(3857)
nepal = nepal.to_crs(3857)
study_districts = study_districts.to_crs(3857)
landslides = landslides.to_crs(3857)

# ======================================
# FIGURE
# ======================================

fig, ax = plt.subplots(
figsize=(12,10)
)

# Study districts
study_districts.plot(
ax=ax,
color='gold',
edgecolor='black',
linewidth=0.8,
alpha=0.7
)

# Nepal boundary
nepal.plot(
ax=ax,
facecolor='none',
edgecolor='black',
linewidth=2
)

# Landslides
landslides.plot(
ax=ax,
color='red',
markersize=3,
alpha=0.6,
label='Landslide Inventory'
)

# Basemap
ctx.add_basemap(
ax,
source=ctx.providers.Esri.WorldShadedRelief,
zoom=7,
attribution=""
)

# Extent
bounds = nepal.total_bounds

ax.set_xlim(
bounds[0],
bounds[2]
)

ax.set_ylim(
bounds[1],
bounds[3]
)

# ======================================
# NORTH ARROW
# ======================================

ax.annotate(
'N',
xy=(0.04,0.90),
xytext=(0.04,0.96),

arrowprops=dict(
facecolor='black',
width=2,
headwidth=8
),

xycoords='axes fraction',

ha='center',
fontsize=12,
fontweight='bold'
)

# ======================================
# SCALE BAR
# ======================================

scalebar = ScaleBar(
0.6,
location='lower center',
pad=0.6,
frameon=True,
color='black',
box_alpha=0.6
)

ax.add_artist(
scalebar
)

# ======================================
# INSET MAP
# ======================================

axins = inset_axes(
ax,

width="45%",
height="45%",

loc='upper right',

bbox_to_anchor=
(0.53,0.53,0.45,0.45),

bbox_transform=ax.transAxes
)

rect = plt.Rectangle(
(0,0),
1,
1,

transform=axins.transAxes,

color='white',
zorder=-1
)

axins.add_patch(rect)

south_asia_list = [

"Nepal",
"India",
"Bangladesh",
"Pakistan",
"Afghanistan",
"Bhutan",
"Sri Lanka"

]

south_asia = world[
world["NAME"].isin(
south_asia_list
)
]

south_asia.plot(
ax=axins,
color='lightgray',
edgecolor='white',
linewidth=0.5
)

nepal.plot(
ax=axins,
color='red'
)

axins.axis('off')

# ======================================
# LEGEND
# ======================================

legend_handles = [

Patch(
color='gold',
label='Study Districts'
),

Patch(
color='red',
label='ICIMOD Landslide Inventory'
)

]

ax.legend(
handles=legend_handles,

loc='lower left',

frameon=True,

fontsize=11,

facecolor='white',

framealpha=0.9
)

ax.axis('off')

plt.savefig(
'/content/Figure1_StudyArea_Landslides.png',
dpi=600,
bbox_inches='tight'
)

plt.show()

In [ ]:
!pip install contextily -q
!pip install matplotlib-scalebar -q

import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx

from matplotlib.patches import Patch
from matplotlib_scalebar.scalebar import ScaleBar
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# ======================================
# LOAD DATA
# ======================================

world = gpd.read_file(
'/content/ne_10m_admin_0_countries_nep.shp'
)

nepal = world[
world["NAME"]=="Nepal"
]

districts = gpd.read_file(
'/content/gadm41_NPL_3.shp'
)

# ======================================
# LOAD LANDSLIDES
# ======================================

landslides = gpd.read_file(
'/content/14dist_ls.shp'
)

# ======================================
# SELECT STUDY DISTRICTS
# ======================================

study_names = [

'Sindhupalchok',
'Sindhuli',
'Ramechhap',
'Okhaldhunga',
'Nuwakot',
'Makwanpur',
'Lalitpur',
'Kavrepalanchok',
'Kathmandu',
'Gorkha',
'Dolakha',
'Dhading',
'Rasuwa'

]

study_districts = districts[
districts['NAME_3'].isin(study_names)
]

# ======================================
# REPROJECT
# ======================================

world = world.to_crs(3857)
nepal = nepal.to_crs(3857)
study_districts = study_districts.to_crs(3857)
landslides = landslides.to_crs(3857)

# ======================================
# FIGURE
# ======================================

fig, ax = plt.subplots(
figsize=(12,10)
)

# Study districts
study_districts.plot(
ax=ax,
color='gold',
edgecolor='black',
linewidth=0.8,
alpha=0.7
)

# Nepal boundary
nepal.plot(
ax=ax,
facecolor='none',
edgecolor='black',
linewidth=2
)

# Landslides
landslides.plot(
ax=ax,
color='red',
markersize=3,
alpha=0.6,
label='Landslide Inventory'
)

# Basemap
ctx.add_basemap(
ax,
source=ctx.providers.Esri.WorldShadedRelief,
zoom=7,
attribution=""
)

# Extent
bounds = nepal.total_bounds

ax.set_xlim(
bounds[0],
bounds[2]
)

ax.set_ylim(
bounds[1],
bounds[3]
)

# ======================================
# NORTH ARROW
# ======================================

ax.annotate(
'N',
xy=(0.04,0.90),
xytext=(0.04,0.96),

arrowprops=dict(
facecolor='black',
width=2,
headwidth=8
),

xycoords='axes fraction',

ha='center',
fontsize=12,
fontweight='bold'
)

# ======================================
# SCALE BAR
# ======================================

scalebar = ScaleBar(
0.6,
location='lower center',
pad=0.6,
frameon=True,
color='black',
box_alpha=0.6
)

ax.add_artist(
scalebar
)

# ======================================
# INSET MAP
# ======================================

axins = inset_axes(
ax,

width="45%",
height="45%",

loc='upper right',

bbox_to_anchor=
(0.53,0.53,0.45,0.45),

bbox_transform=ax.transAxes
)

rect = plt.Rectangle(
(0,0),
1,
1,

transform=axins.transAxes,

color='white',
zorder=-1
)

axins.add_patch(rect)

south_asia_list = [

"Nepal",
"India",
"Bangladesh",
"Pakistan",
"Afghanistan",
"Bhutan",
"Sri Lanka"

]

south_asia = world[
world["NAME"].isin(
south_asia_list
)
]

south_asia.plot(
ax=axins,
color='lightgray',
edgecolor='white',
linewidth=0.5
)

nepal.plot(
ax=axins,
color='red'
)

axins.axis('off')

# ======================================
# LEGEND
# ======================================

legend_handles = [

Patch(
color='gold',
label='Study Districts'
),

Patch(
color='red',
label='ICIMOD Landslide Inventory'
)

]

ax.legend(
handles=legend_handles,

loc='lower left',

frameon=True,

fontsize=11,

facecolor='white',

framealpha=0.9
)

ax.axis('off')

plt.savefig(
'/content/Figure1_StudyArea_Landslides.png',
dpi=600,
bbox_inches='tight'
)

plt.show()

In [ ]:
!pip install contextily -q
!pip install matplotlib-scalebar -q

import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx

from matplotlib.patches import Patch
from matplotlib_scalebar.scalebar import ScaleBar
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# ======================================
# LOAD DATA
# ======================================

world = gpd.read_file(
    '/content/ne_10m_admin_0_countries_nep.shp'
)

nepal = gpd.read_file(
'/content/1_NepalBoundary.shp'
)

districts = gpd.read_file(
    '/content/gadm41_NPL_3.shp'
)

# ======================================
# LOAD ICIMOD LANDSLIDES
# ======================================

landslides = gpd.read_file(
    '/content/14dist_ls.shp'
)

print("Landslide Geometry Types:")
print(landslides.geom_type.unique())

# ======================================
# STUDY DISTRICTS
# ======================================

study_names = [

    'Sindhupalchok',
    'Sindhuli',
    'Ramechhap',
    'Okhaldhunga',
    'Nuwakot',
    'Makwanpur',
    'Lalitpur',
    'Kavrepalanchok',
    'Kathmandu',
    'Gorkha',
    'Dolakha',
    'Dhading',
    'Rasuwa'

]

study_districts = districts[
    districts['NAME_3'].isin(study_names)
]

# ======================================
# REPROJECT TO WEB MERCATOR
# ======================================

world = world.to_crs(3857)
nepal = nepal.to_crs(3857)
study_districts = study_districts.to_crs(3857)
landslides = landslides.to_crs(3857)

# ======================================
# CONVERT LANDSLIDES TO CENTROIDS
# ======================================

landslide_pts = landslides.copy()

landslide_pts["geometry"] = (
    landslide_pts.geometry.centroid
)

print("Number of Landslides:",
      len(landslide_pts))

# ======================================
# CREATE FIGURE
# ======================================

fig, ax = plt.subplots(
    figsize=(12,10)
)

# ======================================
# STUDY DISTRICTS
# ======================================

study_districts.plot(
    ax=ax,
    color='gold',
    edgecolor='black',
    linewidth=0.8,
    alpha=0.70,
    zorder=5
)

# ======================================
# NEPAL OUTLINE
# ======================================

nepal.plot(
    ax=ax,
    facecolor='none',
    edgecolor='black',
    linewidth=2,
    zorder=10
)

# ======================================
# LANDSLIDE INVENTORY
# ======================================

landslide_pts.plot(
    ax=ax,
    color='red',
    markersize=10,
    alpha=0.85,
    zorder=20
)

# ======================================
# BASEMAP
# ======================================

ctx.add_basemap(
    ax,
    source=ctx.providers.Esri.WorldShadedRelief,
    zoom=7,
    attribution=""
)

# ======================================
# EXTENT
# ======================================

bounds = nepal.total_bounds

ax.set_xlim(
    bounds[0],
    bounds[2]
)

ax.set_ylim(
    bounds[1],
    bounds[3]
)

# ======================================
# NORTH ARROW
# ======================================

ax.annotate(
    'N',
    xy=(0.04,0.90),
    xytext=(0.04,0.96),

    arrowprops=dict(
        facecolor='black',
        width=2,
        headwidth=8
    ),

    xycoords='axes fraction',

    ha='center',
    fontsize=12,
    fontweight='bold'
)

# ======================================
# SCALE BAR
# ======================================

scalebar = ScaleBar(
    1,
    units='m',
    dimension='si-length',
    location='lower center',
    frameon=True,
    color='black',
    box_alpha=0.6
)

ax.add_artist(
    scalebar
)

# ======================================
# SOUTH ASIA INSET
# ======================================

axins = inset_axes(
    ax,

    width="45%",
    height="45%",

    loc='upper right',

    bbox_to_anchor=
    (0.53,0.53,0.45,0.45),

    bbox_transform=ax.transAxes
)

rect = plt.Rectangle(
    (0,0),
    1,
    1,

    transform=axins.transAxes,

    color='white',

    zorder=-1
)

axins.add_patch(rect)

south_asia_list = [

    "Nepal",
    "India",
    "Bangladesh",
    "Pakistan",
    "Afghanistan",
    "Bhutan",
    "Sri Lanka"

]

south_asia = world[
    world["NAME"].isin(
        south_asia_list
    )
]

south_asia.plot(
    ax=axins,
    color='lightgray',
    edgecolor='white',
    linewidth=0.5
)

nepal.plot(
    ax=axins,
    color='red'
)

axins.axis('off')

# ======================================
# LEGEND
# ======================================

legend_handles = [

    Patch(
        color='gold',
        label='Study Districts'
    ),

    Patch(
        color='red',
        label='Landslide Inventory'
    )

]

ax.legend(
    handles=legend_handles,

    loc='lower left',

    frameon=True,

    fontsize=11,

    facecolor='white',

    framealpha=0.9
)

# ======================================
# TITLE (OPTIONAL)
# ======================================

# ax.set_title(
#     'Study Area and Landslide Inventory, Nepal',
#     fontsize=14,
#     fontweight='bold'
# )

ax.axis('off')

# ======================================
# SAVE FIGURE
# ======================================

plt.savefig(
    '/content/Figure1_StudyArea_Landslides.png',
    dpi=600,
    bbox_inches='tight'
)

plt.show()